# 02 — Fine-tuning BETO en Colab (T4/A100)

**Dataset:** 7926 tickets limpios (19 clases) · **Modelo:** `dccuchile/bert-base-spanish-wwm-cased`

### Qué hace esta notebook
1. Verifica GPU y instala deps (1 min)
2. Carga `data/processed/` (parquet + mapping) desde Drive o subida manual
3. Split estratificado 80/10/10, `class_weight` para clases raras (Datos n=26)
4. Fine-tuning BETO 5 épocas (T4 ≈ 3 min/época con 256 tokens, ~15 min total)
5. Evalúa val/test (accuracy, macro-F1, reporte por clase) y guarda `ml/models/beto-tickets/`
6. Exporta zip para descargar o guardar en Drive

### Cómo subir los datos a Colab (elige 1)
**Opción A — Drive (recomendado):** en Drive crea `HelpDesk/data/processed/` y sube estos 3 archivos del repo local:
```
data/processed/tickets_clean.parquet  (2.2 MB)
data/processed/tickets_clean.csv      (5.0 MB, fallback)
data/processed/label_mapping.json     (1.6 KB)
```
Luego en Colab monta Drive (celda 2) y aparecen en `/content/drive/MyDrive/HelpDesk/...`

**Opción B — Subida directa:** sin Drive, usa el widget de la celda 2 para subir los 3 archivos (se guardan en `/content/data/processed/`)

**Opción C — Clonar repo:** si el repo es privado, genera un PAT y clona; si es público basta la URL

Estructura esperada dentro del entorno Colab (sea cual sea la opción):
```
/content/                 # o /content/drive/MyDrive/HelpDesk si usas Drive
├── data/processed/
│   ├── tickets_clean.parquet
│   ├── tickets_clean.csv
│   └── label_mapping.json
└── ml/models/beto-tickets/   # se crea al entrenar (config.json, pytorch_model.bin, tokenizer, metrics.json)
```
No necesitas subir `data/raw/` (3.7 MB crudo con PII) ni `data/quarantine/`.

## 0. GPU check + instalar deps

In [ ]:
import torch, platform, sys
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), "| mps", getattr(torch.backends.mps, 'is_available', lambda: False)())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    # Colab T4 = 15 GB, A100 = 40 GB → ambos sobran para BETO+batch16x256
else:
    print("⚠️ Sin GPU: en Colab ve a Entorno > Cambiar tipo de entorno > T4 GPU")

# instala solo si falta (transformers 4.40+ y accelerate). En Colab suele venir torch ya.
!pip -q install transformers==4.44.2 datasets accelerate scikit-learn pyarrow 2>&1 | tail -n 5
import transformers; print("transformers", transformers.__version__)

## 1. Cargar datos: Drive o subida manual

Ejecuta **solo una** de las dos celdas siguientes.

In [ ]:
# --- OPCIÓN A: Drive ---
from pathlib import Path
import os

USE_DRIVE = True  # pon False si prefieres subida manual

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # ajusta si tu carpeta en Drive tiene otro nombre
    ROOT = Path("/content/drive/MyDrive/HelpDesk")
    DATA = ROOT / "data" / "processed"
    OUT  = ROOT / "ml" / "models" / "beto-tickets"
    print(f"ROOT={ROOT}")
    print(f"DATA={DATA} exists={DATA.exists()}")
    if DATA.exists():
        print(list(DATA.iterdir()))
    else:
        print("⚠️ Sube a Drive: HelpDesk/data/processed/{tickets_clean.parquet, tickets_clean.csv, label_mapping.json}")
else:
    print("Usa la celda de subida manual abajo")

In [ ]:
# --- OPCIÓN B: subida manual (si USE_DRIVE=False) ---
from pathlib import Path
from google.colab import files

if not globals().get("USE_DRIVE", True):
    ROOT = Path("/content")
    DATA = ROOT / "data" / "processed"
    OUT  = ROOT / "ml" / "models" / "beto-tickets"
    DATA.mkdir(parents=True, exist_ok=True)
    print(f"Sube los 3 archivos a {DATA} (parquet, csv, json). Se abrirá el selector:")
    uploaded = files.upload()  # selecciona los 3 archivos
    import shutil
    for name, content in uploaded.items():
        dest = DATA / name
        # si el usuario subió con nombres exactos ya está; si no, mueve
        if Path(name).exists():
            shutil.move(name, dest)
        print(f"  -> {dest} ({dest.stat().st_size/1024:.0f} KB)")
    print(list(DATA.iterdir()))
else:
    print("Drive activado, ignora esta celda")

In [ ]:
# --- OPCIÓN C: clonar repo (alternativa a A/B) ---
# Descomenta y ajusta si prefieres clonar en vez de subir solo data/processed
# !git clone https://github.com/TU_ORG/HelpDesk.git /content/HelpDesk
# from pathlib import Path
# ROOT = Path("/content/HelpDesk")
# DATA = ROOT / "data" / "processed"
# OUT  = ROOT / "ml" / "models" / "beto-tickets"
# print(list(DATA.iterdir()))

## 2. Verificar datos + hyperparams

In [ ]:
import json, pandas as pd

# resuelve ROOT/DATA si no se definieron arriba (por reinicio de kernel)
from pathlib import Path
if 'ROOT' not in globals():
    # intenta autodetectar
    for cand in [Path("/content/drive/MyDrive/HelpDesk"), Path("/content/HelpDesk"), Path("/content")]:
        if (cand / "data" / "processed" / "label_mapping.json").exists():
            ROOT = cand; break
    else:
        ROOT = Path("/content")
    DATA = ROOT / "data" / "processed"
    OUT  = ROOT / "ml" / "models" / "beto-tickets"

parquet = DATA / "tickets_clean.parquet"
csv = DATA / "tickets_clean.csv"
mapping_p = DATA / "label_mapping.json"

if parquet.exists():
    df = pd.read_parquet(parquet)
else:
    df = pd.read_csv(csv)
with open(mapping_p, encoding="utf-8") as f:
    mp = json.load(f)
id2label = {int(k): v for k, v in mp["id2label"].items()}
label2id = {v: k for k, v in id2label.items()}

print(f"filas={len(df)} cols={list(df.columns)}")
print(f"clases={len(id2label)} -> {list(id2label.values())[:5]} ...")
print(df["label_id"].value_counts().sort_index().to_string())
print(f"\ntexto ejemplo (60ch): {df['texto'].iloc[0][:120]}...")
print(f"\nOUT -> {OUT}")

# hyperparams — ajusta aquí antes de entrenar
HP = dict(
    model="dccuchile/bert-base-spanish-wwm-cased",
    epochs=5,
    batch=16,          # T4: 16×256 cabe (~7 GB). Si OOM baja a 8
    eval_batch=32,
    lr=2e-5,
    max_length=256,    # p95=256, p99~320. Usa 128 para ir más rápido (~30% menos tiempo)
    weight_decay=0.01,
    seed=42,
)
print(HP)
# smoke test rápido en Colab (descomenta para probar 1 época con 1500 filas):
# HP["epochs"]=1; SUBSET=1500
SUBSET=0  # 0 = full 7926

## 3. Entrenar BETO (con `class_weight` + early stopping)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# split 80/10/10 estratificado
idx = np.arange(len(df))
if SUBSET and SUBSET < len(df):
    df_sub = df.sample(n=SUBSET, random_state=HP["seed"]).reset_index(drop=True)
else:
    df_sub = df
print(f"usando {len(df_sub)} filas")

tr_idx, tmp_idx = train_test_split(idx[:len(df_sub)], test_size=0.20, random_state=HP["seed"], stratify=df_sub["label_id"])
val_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, random_state=HP["seed"], stratify=df_sub.iloc[tmp_idx]["label_id"])
df_train, df_val, df_test = df_sub.iloc[tr_idx].reset_index(drop=True), df_sub.iloc[val_idx].reset_index(drop=True), df_sub.iloc[te_idx].reset_index(drop=True)
print(f"train {len(df_train)} val {len(df_val)} test {len(df_test)}")

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

tokenizer = AutoTokenizer.from_pretrained(HP["model"])

class TicketDS(torch.utils.data.Dataset):
    def __init__(self, d):
        self.enc = tokenizer(d["texto"].tolist(), truncation=True, padding="max_length", max_length=HP["max_length"])
        self.labels = d["label_id"].astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

train_ds, val_ds, test_ds = TicketDS(df_train), TicketDS(df_val), TicketDS(df_test)

# class weights (Datos n=26 pesa ~80× vs Piezas gráficas n=2087)
label_ids_sorted = sorted(np.unique(df["label_id"].values))
weights = compute_class_weight("balanced", classes=np.array(label_ids_sorted), y=df_train["label_id"].values)
weight_map = {lid: w for lid, w in zip(label_ids_sorted, weights)}
print(f"weights max {max(weights):.1f} ejemplo {list(weight_map.items())[:3]}")

model = AutoModelForSequenceClassification.from_pretrained(HP["model"], num_labels=len(id2label), id2label=id2label, label2id=label2id)

def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro", zero_division=0), "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0)}

OUT.mkdir(parents=True, exist_ok=True)
args = TrainingArguments(
    output_dir=str(OUT),
    num_train_epochs=HP["epochs"],
    per_device_train_batch_size=HP["batch"],
    per_device_eval_batch_size=HP["eval_batch"],
    learning_rate=HP["lr"],
    weight_decay=HP["weight_decay"],
    warmup_steps=50,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    seed=HP["seed"],
    report_to="none",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),  # acelera en T4/A100, en CPU se ignora
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        import torch.nn.functional as F
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        w = torch.tensor([weight_map.get(lid, 1.0) for lid in sorted(label2id.values())], dtype=torch.float, device=logits.device)
        loss = F.cross_entropy(logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

In [ ]:
# eval + reporte
import json
val_m = trainer.evaluate(val_ds)
test_m = trainer.evaluate(test_ds)
print({"val": val_m, "test": test_m})

pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=1)
y_true = pred.label_ids
labels_present = sorted(set(int(x) for x in y_true) | set(int(x) for x in y_pred))
print(classification_report(y_true, y_pred, labels=labels_present, target_names=[id2label[i] for i in labels_present], zero_division=0))

# guardar
trainer.save_model(str(OUT))
tokenizer.save_pretrained(str(OUT))
with open(OUT / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({"val": val_m, "test": test_m, "hp": HP, "clases": len(id2label)}, f, ensure_ascii=False, indent=2)
with open(OUT / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}}, f, ensure_ascii=False, indent=2)
print(f"guardado en {OUT}")
!ls -lh "$OUT" 2>&1 | head -n 20
!cat "$OUT/metrics.json"

## 4. Exportar / descargar modelo

El modelo ya está en `OUT` (Drive si usaste Drive, `/content/ml/models/...` si subida manual). Elige cómo llevártelo:

In [ ]:
from pathlib import Path
import shutil

# zip para descargar (funciona en ambos modos)
zip_path = Path("/content/beto-tickets.zip")
if OUT.exists():
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", OUT)
    print(f"zip {zip_path} -> {zip_path.stat().st_size/1e6:.1f} MB")
    # en Drive el OUT ya está persistido; el zip es solo para descargar rápido
    if "drive" in str(OUT):
        drive_zip = OUT.parent / "beto-tickets.zip"
        shutil.copy(zip_path, drive_zip)
        print(f"copiado a Drive: {drive_zip}")
else:
    print(f"no existe {OUT}")

# botón de descarga (solo funciona si ejecutas en colab con subida manual o si quieres bajar el zip de Drive)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print(e)
    print(f"Descarga manual: Archivos > {zip_path}")

In [ ]:
# (opcional) subir el zip de vuelta al repo / release
# Una vez descargado beto-tickets.zip, en tu máquina local:
#   unzip beto-tickets.zip -d ml/models/beto-tickets
#   # o súbelo como artefacto a Hugging Face:
#   # huggingface-cli upload tu_usuario/beto-helpdesk ml/models/beto-tickets
print("Listo. El modelo NO se commitea (está en .gitignore: ml/models/). Súbelo a Drive/HF o descomprime localmente.")

## 5. Prueba rápida de inferencia (opcional)

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=str(OUT), tokenizer=str(OUT), device=0 if torch.cuda.is_available() else -1)
for txt in [
    "Solicito diseño de pieza gráfica para cartelera institucional",
    "El computador no enciende y hace ruido extraño al arrancar",
    "Necesito restablecer contraseña del correo institucional",
]:
    print(txt, "->", clf(txt, truncation=True)[0])